# FDC-Monitoring LangGraph RAG Agent

**기존 `agentic_rag.ipynb`를 보존하면서**, LangGraph의 조건부 분기를 활용해
VOC 응답 흐름을 **노드 기반 그래프**로 구조화한 확장 버전입니다.

---

## 그래프 구조

```
START
  └─► extract_entities
         └─► validate_entities
                ├─[invalid_entity]──► refusal_answer ──► END
                ├─[need_clarification]► clarification_answer ──► END
                └─[valid_query]──► retrieve_context
                                       └─► generate_answer
                                              └─► validate_citation ──► END
```

## 각 노드 역할

| 노드 | 역할 |
|---|---|
| `extract_entities` | 질문에서 alarm_code / equipment_id / intent 추출 |
| `validate_entities` | master CSV 대조 → invalid / clarification / valid 결정 |
| `retrieve_context` | FAISS retriever로 관련 section chunk 검색 |
| `generate_answer` | 검색 근거 기반 6섹션 답변 생성 |
| `validate_citation` | 답변 속 [SECTION-ID]가 검색 컨텍스트에 존재하는지 검증 |
| `refusal_answer` | 존재하지 않는 코드·설비 또는 범위 밖 요청 거절 |
| `clarification_answer` | 부족한 정보를 되묻는 명확화 요청 |

## 기존 MVP와의 차이

| | `agentic_rag.ipynb` (MVP) | `langgraph_rag_agent.ipynb` (이번) |
|---|---|---|
| 흐름 제어 | ReAct loop (LLM이 자유 판단) | 명시적 노드 그래프 |
| 엔티티 검증 | LLM 프롬프트에 의존 | master CSV 대조 |
| 분기 처리 | LLM이 암묵적 처리 | 조건부 엣지로 명시 |
| 인용 검증 | 없음 | validate_citation 노드 |
| 추적·디버깅 | LangSmith trace | 노드별 상태 덤프 가능 |

## 0. (필요 시) 의존성 설치

In [ ]:
# %pip install -q langchain langchain-openai langchain-community langchain-text-splitters
# %pip install -q faiss-cpu python-dotenv langgraph

## 1. 환경 변수 로드 + API 검증

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

env_path = PROJECT_ROOT / ".env"
loaded = load_dotenv(dotenv_path=env_path, override=True)
print(f".env 로드: {loaded}  (경로: {env_path})")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

def _mask(k):
    if not k:
        return "(없음)"
    return k[:6] + "..." + k[-4:] if len(k) > 12 else "***"

print(f"OPENAI_API_KEY    : {_mask(OPENAI_API_KEY)}")
print(f"LANGSMITH_API_KEY : {_mask(LANGSMITH_API_KEY)}")

assert OPENAI_API_KEY, "OPENAI_API_KEY 가 .env 에 없습니다."

if LANGSMITH_API_KEY:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
    os.environ.setdefault("LANGCHAIN_PROJECT", "fdc-langgraph-rag")
    print("LangSmith tracing 활성화 — project=fdc-langgraph-rag")
else:
    print("LangSmith 키 없음 — 트레이싱 비활성화")

## 2. 매뉴얼 로드 + section chunk 생성

`agentic_rag.ipynb`와 동일한 로직을 재사용합니다.

In [ ]:
MANUAL_DIR = PROJECT_ROOT / "data" / "manuals"
MANUAL_FILES = [
    "alarm_code_guide.md",
    "troubleshooting_guide.md",
    "operation_policy.md",
    "system_user_manual.md",
    "faq.md",
]

raw_manuals = {}
for fname in MANUAL_FILES:
    path = MANUAL_DIR / fname
    text = path.read_text(encoding="utf-8")
    raw_manuals[fname] = text
    print(f"  [{fname}] {len(text):,} chars")

In [ ]:
import re
from langchain_core.documents import Document

HEADER_RE = re.compile(
    r"^(?P<hashes>#{2,3})\s+(?P<sid>[A-Z]+-[A-Z0-9-]+)\s*\|\s*(?P<title>.+?)\s*$",
    re.MULTILINE,
)


def split_by_section(file_name: str, text: str):
    matches = list(HEADER_RE.finditer(text))
    if not matches:
        return []
    docs = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].rstrip()
        sid = m.group("sid")
        title = m.group("title").strip()
        citation = f"[{sid}] {title}"
        if len(citation) > 60:
            citation = citation[:57] + "..."
        docs.append(
            Document(
                page_content=body,
                metadata={
                    "file_name": file_name,
                    "section_id": sid,
                    "title": title,
                    "citation": citation,
                },
            )
        )
    return docs


chunks = []
for fname, text in raw_manuals.items():
    chunks.extend(split_by_section(fname, text))

print(f"총 chunk 수: {len(chunks)}")
from collections import Counter
for fname, n in Counter(c.metadata["file_name"] for c in chunks).items():
    print(f"  {fname}: {n}")

## 3. FAISS Vectorstore + Retriever

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
vectordb = FAISS.from_documents(chunks, embedding)
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

print(f"FAISS index 생성 완료 — 벡터 {vectordb.index.ntotal}개")

# Sanity check
_test = retriever.invoke("TEMP-H-001 챔버 과열 임계치")
print("Sanity check (TEMP-H-001 검색):")
for i, d in enumerate(_test[:2], 1):
    print(f"  [{i}] {d.metadata['citation']}")

## 4. Master 데이터 로드

`validate_entities` 노드에서 알람 코드와 설비 ID의 존재 여부를 검증하는 데 사용합니다.

In [ ]:
import csv

ALARM_MASTER_PATH = PROJECT_ROOT / "data" / "db" / "alarm_code_master.csv"
EQUIP_MASTER_PATH = PROJECT_ROOT / "data" / "db" / "equipment_master.csv"

alarm_codes_valid: set[str] = set()
with open(ALARM_MASTER_PATH, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        alarm_codes_valid.add(row["alarm_code"].strip())

equipment_ids_valid: set[str] = set()
with open(EQUIP_MASTER_PATH, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        equipment_ids_valid.add(row["equipment_id"].strip())

print(f"등록 알람 코드: {len(alarm_codes_valid)}개  (예: {sorted(alarm_codes_valid)[:4]})")
print(f"등록 설비 ID:   {len(equipment_ids_valid)}개  (예: {sorted(equipment_ids_valid)[:4]})")

## 5. LangGraph State 정의

`VOCState`는 그래프의 모든 노드가 공유하는 상태 객체입니다.  
각 노드는 `dict`를 반환해 해당 키만 업데이트하며, 나머지 키는 유지됩니다.

In [ ]:
from typing import Optional, Literal
from typing_extensions import TypedDict


class VOCState(TypedDict):
    """그래프 전체를 흐르는 상태 스키마."""

    # ── 입력
    question: str

    # ── extract_entities 결과
    alarm_code: Optional[str]     # 추출된 알람 코드 (없으면 None)
    equipment_id: Optional[str]   # 추출된 설비 ID (없으면 None)
    intent: Optional[str]         # alarm_inquiry / sop_inquiry / policy_inquiry /
                                  # symptom / system_usage / out_of_scope / unknown

    # ── validate_entities 결과
    validation_status: Optional[Literal["valid_query", "invalid_entity", "need_clarification"]]
    validation_reason: Optional[str]

    # ── retrieve_context 결과
    retrieved_context: Optional[str]
    retrieved_section_ids: Optional[list]

    # ── generate_answer 결과
    answer: Optional[str]

    # ── validate_citation 결과
    cited_section_ids: Optional[list]
    citation_valid: Optional[bool]

    # ── 최종 출력
    final_answer: Optional[str]


print("VOCState 정의 완료")
print(f"필드: {list(VOCState.__annotations__.keys())}")

## 6. 노드 구현

### 6-1. `extract_entities` — 엔티티 추출

In [ ]:
import json as _json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

_EXTRACT_TMPL = """다음 질문에서 아래 정보를 JSON으로 추출하세요.

질문: {question}

추출 항목:
- alarm_code: 알람 코드 (예: TEMP-H-001, PRES-C-001). 언급 없으면 null
- equipment_id: 설비 ID (예: ETCH-DRY-01, THIN-CVD-02). 언급 없으면 null
- intent: 다음 중 하나
  * "alarm_inquiry"  — 알람 코드 의미·임계치 문의
  * "sop_inquiry"    — SOP·조치 절차 문의
  * "policy_inquiry" — 운영 정책·권한·escalation 문의
  * "system_usage"   — 시스템 사용법 문의
  * "symptom"        — 증상만 서술하고 코드 없음
  * "out_of_scope"   — FDC-Monitoring 범위 밖 질문
  * "unknown"        — 분류 불가

JSON 외 텍스트 없이 순수 JSON만 출력하세요.
예시: {{"alarm_code": "TEMP-H-001", "equipment_id": "ETCH-DRY-01", "intent": "alarm_inquiry"}}"""


def extract_entities(state: VOCState) -> dict:
    """질문에서 alarm_code, equipment_id, intent를 LLM으로 추출."""
    response = llm.invoke(_EXTRACT_TMPL.format(question=state["question"]))

    try:
        text = response.content.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        extracted = _json.loads(text.strip())
    except Exception:
        extracted = {"alarm_code": None, "equipment_id": None, "intent": "unknown"}

    return {
        "alarm_code": extracted.get("alarm_code"),
        "equipment_id": extracted.get("equipment_id"),
        "intent": extracted.get("intent", "unknown"),
    }


# 단위 테스트
_sample = {"question": "ETCH-DRY-01 CH-A에서 TEMP-H-001 알람이 떴습니다. 의미가 뭔가요?"}
print("extract_entities 단위 테스트:")
print(extract_entities(_sample))

### 6-2. `validate_entities` — Master 데이터 대조 + routing 결정

In [ ]:
def validate_entities(state: VOCState) -> dict:
    """alarm_code / equipment_id를 master CSV와 대조해 routing 상태를 결정."""

    alarm_code = state.get("alarm_code")
    equipment_id = state.get("equipment_id")
    intent = state.get("intent", "unknown")

    # 1) FDC-Monitoring 범위 밖
    if intent == "out_of_scope":
        return {
            "validation_status": "invalid_entity",
            "validation_reason": (
                "요청하신 내용은 FDC-Monitoring 시스템의 운영 범위를 벗어납니다. "
                "관련 담당 팀에 문의해 주세요."
            ),
        }

    # 2) 알람 코드가 명시됐지만 master에 없음
    if alarm_code and alarm_code not in alarm_codes_valid:
        return {
            "validation_status": "invalid_entity",
            "validation_reason": (
                f"알람 코드 '{alarm_code}'는 등록된 코드가 아닙니다. "
                "정확한 코드를 다시 확인해 주세요."
            ),
        }

    # 3) 설비 ID가 명시됐지만 master에 없음
    if equipment_id and equipment_id not in equipment_ids_valid:
        return {
            "validation_status": "invalid_entity",
            "validation_reason": (
                f"설비 ID '{equipment_id}'는 등록된 설비가 아닙니다. "
                "설비 ID를 다시 확인해 주세요."
            ),
        }

    # 4) 증상만 있고 alarm_code 미확인 → 명확화 필요
    if intent == "symptom" and not alarm_code:
        return {
            "validation_status": "need_clarification",
            "validation_reason": (
                "증상은 파악됐지만 알람 코드가 없어 정확한 원인을 특정하기 어렵습니다."
            ),
        }

    # 5) 의도가 alarm/sop 인데 코드·설비 모두 없음 → 명확화 필요
    if intent in ("alarm_inquiry", "sop_inquiry") and not alarm_code and not equipment_id:
        return {
            "validation_status": "need_clarification",
            "validation_reason": "알람 코드 또는 설비 ID 정보가 부족합니다.",
        }

    # 6) 분류 불가 → 명확화
    if intent == "unknown":
        return {
            "validation_status": "need_clarification",
            "validation_reason": "질문 의도를 파악하기 어렵습니다.",
        }

    # 7) 그 외 → 유효 쿼리
    return {
        "validation_status": "valid_query",
        "validation_reason": None,
    }


print("validate_entities 정의 완료")

### 6-3. `route_after_validation` — 조건부 엣지 라우팅 함수

LangGraph에서 조건부 분기는 **노드가 아닌 라우팅 함수**로 구현합니다.  
`add_conditional_edges(source_node, routing_fn, mapping)` 형태로 등록하면  
그래프가 `validate_entities` 이후 3방향으로 분기됩니다.

In [ ]:
def route_after_validation(state: VOCState) -> str:
    """validate_entities 결과를 읽어 다음 노드 이름을 반환."""
    status = state.get("validation_status")
    if status == "invalid_entity":
        return "refusal_answer"
    elif status == "need_clarification":
        return "clarification_answer"
    else:
        return "retrieve_context"


print("route_after_validation 정의 완료")

### 6-4. RAG 경로 노드: `retrieve_context` / `generate_answer` / `validate_citation`

In [ ]:
def retrieve_context(state: VOCState) -> dict:
    """FAISS retriever로 관련 section chunk 검색."""

    alarm_code = state.get("alarm_code") or ""
    question = state["question"]
    search_query = f"{alarm_code} {question}".strip() if alarm_code else question

    docs = retriever.invoke(search_query)

    if not docs:
        return {
            "retrieved_context": "관련 매뉴얼 항목을 찾지 못했습니다.",
            "retrieved_section_ids": [],
        }

    section_ids = [d.metadata["section_id"] for d in docs]
    blocks = []
    for d in docs:
        block = (
            f"### {d.metadata['citation']}\n"
            f"(출처: {d.metadata['file_name']})\n\n"
            f"{d.page_content}"
        )
        blocks.append(block)

    return {
        "retrieved_context": "\n\n---\n\n".join(blocks),
        "retrieved_section_ids": section_ids,
    }

In [ ]:
_GENERATE_TMPL = """너는 반도체 FDC-Monitoring 시스템의 VOC 응답 전문가다.
아래 매뉴얼 검색 결과를 근거로, 사용자 질문에 대한 구조화된 한국어 답변을 작성하라.

==== 검색된 매뉴얼 컨텍스트 ====
{context}

==== 사용자 질문 ====
{question}

==== 답변 형식 (6섹션) ====

**[1] 핵심 요약**
(1~2 문장으로 핵심만)

**[2] 알람 / 문제 정의**
(코드 의미, 발생 조건·임계치)

**[3] 원인 분석**
(주요 원인 bullet)

**[4] 조치 절차**
(단계별 번호 매기기)

**[5] 관련 정책 및 주의사항**
(SLA, 권한, escalation 등)

**[6] 인용 출처**
(참조한 section_id를 [SECTION-ID] 형식으로 나열)

중요:
- 컨텍스트에 없는 내용은 추측하지 말 것
- [6] 인용 출처는 검색 컨텍스트에 실제로 있는 section_id만 사용할 것
- 근거를 못 찾으면 [6]에 "해당 정보 없음"이라 명시"""


def generate_answer(state: VOCState) -> dict:
    """검색 근거 기반 6섹션 답변 생성."""
    context = state.get("retrieved_context", "컨텍스트 없음")
    question = state["question"]

    response = llm.invoke(_GENERATE_TMPL.format(context=context, question=question))
    return {"answer": response.content}

In [ ]:
def validate_citation(state: VOCState) -> dict:
    """답변 속 [SECTION-ID] 인용이 검색 컨텍스트에 실제로 존재하는지 검증."""

    answer = state.get("answer", "")
    retrieved_ids = set(state.get("retrieved_section_ids") or [])

    # 답변에서 [SECTION-ID] 패턴 추출
    cited_ids = re.findall(r"\[([A-Z]+-[A-Z0-9-]+)\]", answer)
    cited_ids = list(dict.fromkeys(cited_ids))  # 중복 제거, 순서 유지

    invalid_citations = [sid for sid in cited_ids if sid not in retrieved_ids]
    citation_valid = len(invalid_citations) == 0

    if citation_valid:
        final_answer = answer
    else:
        warning = (
            f"\n\n---\n> **[인용 검증 경고]** "
            f"다음 section_id는 검색 컨텍스트에서 확인되지 않았습니다: "
            f"{', '.join(invalid_citations)}"
        )
        final_answer = answer + warning

    return {
        "cited_section_ids": cited_ids,
        "citation_valid": citation_valid,
        "final_answer": final_answer,
    }


print("RAG 경로 노드 3종 정의 완료")

### 6-5. 종단 노드: `refusal_answer` / `clarification_answer`

In [ ]:
def refusal_answer(state: VOCState) -> dict:
    """존재하지 않는 코드·설비 또는 범위 밖 요청에 대한 정중한 거절 생성."""

    reason = state.get("validation_reason", "처리할 수 없는 요청입니다.")
    alarm_code = state.get("alarm_code")
    equipment_id = state.get("equipment_id")

    prompt = f"""다음 상황에서 정중하고 명확한 거절 응답을 한국어로 작성하라.

상황: {reason}
언급된 알람 코드: {alarm_code or '없음'}
언급된 설비 ID:   {equipment_id or '없음'}
원래 질문: {state['question']}

안내 사항:
- 어떤 문제인지 구체적으로 설명하라
- 사용자가 다음에 무엇을 해야 하는지 안내하라
- 2~3 문장으로 간결하게"""

    response = llm.invoke(prompt)
    return {
        "final_answer": response.content,
        "citation_valid": None,
    }


def clarification_answer(state: VOCState) -> dict:
    """부족한 정보를 되묻는 명확화 요청 응답 생성."""

    reason = state.get("validation_reason", "정보가 부족합니다.")
    intent = state.get("intent", "unknown")

    prompt = f"""다음 상황에서 사용자에게 추가 정보를 요청하는 응답을 한국어로 작성하라.

상황: {reason}
파악된 의도: {intent}
원래 질문: {state['question']}

요청할 정보 (해당되는 것만):
- 설비 ID (예: ETCH-DRY-01, THIN-CVD-02)
- 알람 코드 (예: TEMP-H-001, PRES-C-001)
- 증상 발생 시점·빈도
- 챔버 번호 (해당 시)

친근하고 도움이 되는 톤으로, 3~4 문장 이내로 작성하라."""

    response = llm.invoke(prompt)
    return {
        "final_answer": response.content,
        "citation_valid": None,
    }


print("refusal_answer / clarification_answer 정의 완료")

## 7. 그래프 조립 + 컴파일

`StateGraph`에 노드를 등록하고 엣지를 연결합니다.  
`add_conditional_edges`가 LangGraph 조건부 분기의 핵심입니다.

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(VOCState)

# ── 노드 등록
workflow.add_node("extract_entities", extract_entities)
workflow.add_node("validate_entities", validate_entities)
workflow.add_node("retrieve_context", retrieve_context)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("validate_citation", validate_citation)
workflow.add_node("refusal_answer", refusal_answer)
workflow.add_node("clarification_answer", clarification_answer)

# ── 직선 엣지
workflow.add_edge(START, "extract_entities")
workflow.add_edge("extract_entities", "validate_entities")
workflow.add_edge("retrieve_context", "generate_answer")
workflow.add_edge("generate_answer", "validate_citation")
workflow.add_edge("validate_citation", END)
workflow.add_edge("refusal_answer", END)
workflow.add_edge("clarification_answer", END)

# ── 조건부 분기: validate_entities → route_after_validation → 3방향
workflow.add_conditional_edges(
    "validate_entities",          # 분기 시작 노드
    route_after_validation,        # 라우팅 함수 (상태를 읽어 노드 이름 반환)
    {
        "retrieve_context":     "retrieve_context",
        "refusal_answer":       "refusal_answer",
        "clarification_answer": "clarification_answer",
    }
)

# ── 컴파일
app = workflow.compile()
print("그래프 컴파일 완료")

## 8. 그래프 시각화 (Mermaid)

LangGraph는 그래프 구조를 Mermaid 다이어그램으로 자동 변환합니다.

In [ ]:
# Mermaid 소스 출력
mermaid_src = app.get_graph().draw_mermaid()
print("=== Mermaid Source ===")
print(mermaid_src)

In [ ]:
# PNG 렌더링 (mermaid.ink API 사용 — 인터넷 필요)
from IPython.display import Image, display

try:
    img_bytes = app.get_graph().draw_mermaid_png()
    display(Image(img_bytes))
except Exception as e:
    print(f"PNG 렌더링 실패: {e}")
    print("위 Mermaid 소스를 https://mermaid.live 에 붙여넣어 시각화하세요.")

### 수동 Mermaid 다이어그램 (PNG 실패 시 참조)

```mermaid
flowchart TD
    START([START]) --> EX[extract_entities]
    EX --> VA[validate_entities]
    VA -->|invalid_entity| RA[refusal_answer]
    VA -->|need_clarification| CA[clarification_answer]
    VA -->|valid_query| RC[retrieve_context]
    RC --> GA[generate_answer]
    GA --> VC[validate_citation]
    VC --> END_V([END])
    RA --> END_R([END])
    CA --> END_C([END])

    style RA fill:#ffcccc
    style CA fill:#fff3cc
    style VC fill:#ccffcc
    style RC fill:#cce5ff
    style GA fill:#cce5ff
```

## 9. End-to-End 테스트

### 4가지 경로 커버리지

| 테스트 | 기대 경로 | VOC |
|---|---|---|
| Easy (valid) | valid_query → RAG → validate_citation | VOC-2026-0001 |
| Trap (invalid code) | invalid_entity → refusal | VOC-2026-0027 |
| Symptom (no code) | need_clarification → clarification | VOC-2026-0017 |
| Medium SOP | valid_query → RAG → validate_citation | VOC-2026-0009 |

In [ ]:
import json as _json_voc

VOC_PATH = PROJECT_ROOT / "data" / "voc" / "voc_samples.json"
all_vocs = _json_voc.loads(VOC_PATH.read_text(encoding="utf-8"))["vocs"]
voc_by_id = {v["voc_id"]: v for v in all_vocs}

print(f"VOC 전체 {len(all_vocs)}건 로드")


def run_graph(voc_id: str, verbose: bool = True) -> VOCState:
    """단일 VOC를 그래프에 실행하고 최종 상태를 반환."""
    voc = voc_by_id[voc_id]
    question = voc["content"]

    SEP = "=" * 80
    if verbose:
        print(SEP)
        print(f"[{voc_id}] difficulty={voc['difficulty']} | category={voc['category']}")
        print(f"Q: {question}")
        print("-" * 80)

    # 그래프 실행 (stream으로 노드별 상태 덤프)
    final_state = None
    for step in app.stream({"question": question}, stream_mode="updates"):
        for node_name, update in step.items():
            if verbose:
                # 각 노드가 업데이트한 키만 보여주기
                keys = list(update.keys())
                print(f"  ▶ [{node_name}] 업데이트: {keys}")
                if "validation_status" in update:
                    print(f"       validation_status = {update['validation_status']}")
                    if update.get("validation_reason"):
                        print(f"       reason = {update['validation_reason']}")
                if "retrieved_section_ids" in update:
                    print(f"       retrieved = {update['retrieved_section_ids']}")
                if "cited_section_ids" in update:
                    print(f"       cited    = {update['cited_section_ids']}")
                    print(f"       valid    = {update.get('citation_valid')}")

    # 전체 최종 상태 가져오기
    final_state = app.invoke({"question": question})

    if verbose:
        print("-" * 80)
        print("[최종 답변]")
        for line in (final_state.get("final_answer") or "").splitlines():
            print(f"  {line}")
        print(SEP)
        print()

    return final_state

### 테스트 1 — Easy (유효한 알람 코드 + 설비 → RAG 경로)

In [ ]:
state1 = run_graph("VOC-2026-0001")  # ETCH-DRY-01 / TEMP-H-001

### 테스트 2 — Trap (존재하지 않는 알람 코드 → refusal 경로)

In [ ]:
state2 = run_graph("VOC-2026-0027")  # TEMP-Z-999 (등록되지 않은 코드)

### 테스트 3 — 증상만 있고 코드 없음 → clarification 경로

In [ ]:
state3 = run_graph("VOC-2026-0017")  # 챔버 과열 증상 (코드 미명시)

### 테스트 4 — Medium SOP (COMM-H-001 EAP 통신 단절 → RAG + validate_citation)

In [ ]:
state4 = run_graph("VOC-2026-0013")  # COMM-H-001 EAP 통신 단절

## 10. 상태 인스펙션 — 노드별 중간값 확인

LangGraph의 장점: `stream_mode="values"`로 각 노드 실행 후 전체 상태를 snapshot처럼 볼 수 있습니다.

In [ ]:
# RAG 경로의 각 노드 실행 후 상태 스냅샷
voc_inspect = voc_by_id["VOC-2026-0009"]  # TEMP-H-001 SOP 문의

print(f"Q: {voc_inspect['content']}")
print()

snapshots = []
for snapshot in app.stream(
    {"question": voc_inspect["content"]},
    stream_mode="values",
):
    snapshots.append(dict(snapshot))

# 각 스냅샷에서 핵심 필드만 출력
SHOW_KEYS = [
    "alarm_code", "equipment_id", "intent",
    "validation_status", "retrieved_section_ids",
    "cited_section_ids", "citation_valid",
]

for i, snap in enumerate(snapshots):
    print(f"--- Snapshot {i} ---")
    for k in SHOW_KEYS:
        if k in snap and snap[k] is not None:
            print(f"  {k}: {snap[k]}")
    print()

## 정리 및 다음 단계

### 이번 노트북에서 구현한 것

| 항목 | 방식 |
|---|---|
| 엔티티 추출 | LLM + JSON 파싱 |
| Master 검증 | CSV set 대조 |
| 조건부 분기 | `add_conditional_edges` (3방향) |
| RAG 검색 | FAISS + section chunk |
| 답변 생성 | 6섹션 구조 프롬프트 |
| 인용 검증 | 정규식 → set 교차 검사 |
| 시각화 | Mermaid (자동 생성) |

### 다음 단계 후보

1. **멀티턴 루프**: `validate_citation` 실패 시 `generate_answer`로 되돌아가는 retry 엣지
2. **BM25 하이브리드**: dense + sparse 검색 앙상블
3. **LLM-as-Judge**: 답변 품질 자동 채점 노드 추가
4. **evaluation 연동**: `eval_groundtruth.json`의 gold_docs recall 측정
5. **Human-in-the-loop**: `clarification_answer` 후 사용자 응답을 받아 그래프 재진입